# Class-Conditional VAE from Scratch (Colab)

Companion notebook for the [Class-Conditional VAE](https://overfitting.club/posts/tutorials/deep_learning/class_conditional_vae/class_conditional_vae.html) tutorial. We implement an **adaLN cVAE** — a class-conditional variational autoencoder that uses Adaptive Layer Normalization to inject class information at every layer, the same mechanism used by DiT for diffusion models.

See the full tutorial for theory, diagrams, and the spatial concat comparison.

**What you'll build:**
- A conditional VAE that generates specific garment types on demand
- Interactive widgets for class blending, interpolation, and disentanglement
- The adaLN conditioning mechanism used by state-of-the-art diffusion models

In [ ]:
#@title Install dependencies
try:
    import torch, plotly, rich, sklearn
except ImportError:
    !pip install -q torch torchvision plotly ipywidgets rich scikit-learn tensorboard

### Hyperparameters

| Parameter | What it controls |
|-----------|------------------|
| `BATCH_SIZE` | Images per training step |
| `EPOCHS` | Full passes over the 60k training images |
| `KL_WEIGHT` | β=0.01 — higher than the unconditional VAE (β=0.0005) because generation is now the primary goal. See [A higher β for generation](https://overfitting.club/posts/tutorials/deep_learning/class_conditional_vae/class_conditional_vae.html#a-higher-β-for-generation) |
| `Z_CHANNELS` | Channels in the spatial latent (z_ch × 7 × 7) |
| `COND_DIM` | Dimension of the class embedding vector |

In [ ]:
#@title Configuration { run: "auto" }

BATCH_SIZE = 256  #@param {type:"integer"}
EPOCHS = 200  #@param {type:"integer"}
KL_WEIGHT = 0.01  #@param {type:"number"}
LEARNING_RATE = 1e-3  #@param {type:"number"}
Z_CHANNELS = 1  #@param {type:"integer"}
COND_DIM = 128  #@param {type:"integer"}
NUM_CLASSES = 10

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from rich.console import Console
from rich.table import Table

console = Console()
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
console.print(f"[bold green]Device:[/bold green] {DEVICE}")

writer = SummaryWriter("runs/adaln_cvae")

## 1. Dataset: FashionMNIST

Same dataset as the [autoencoder tutorial](https://overfitting.club/posts/tutorials/deep_learning/autoencoder/autoencoder.html), but now we use the class labels for conditioning.

In [ ]:
#@title Load FashionMNIST

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

t = Table(title="FashionMNIST Dataset")
t.add_column("Split", style="cyan")
t.add_column("Samples", style="green")
t.add_column("Image Size", style="magenta")
t.add_column("Classes", style="dim")
t.add_row("Train", str(len(train_dataset)), "28 \u00d7 28 \u00d7 1", "10")
t.add_row("Test", str(len(test_dataset)), "28 \u00d7 28 \u00d7 1", "10")
console.print(t)

# Fixed test batch for visualizations
sample_images, sample_labels = next(iter(test_loader))

# Sample grid
indices = []
for c in range(10):
    class_idx = (sample_labels == c).nonzero(as_tuple=True)[0][:2]
    indices.extend(class_idx.tolist())
indices = indices[:20]

fig = make_subplots(
    rows=2, cols=10,
    subplot_titles=[CLASS_NAMES[sample_labels[i].item()] for i in indices],
    vertical_spacing=0.08, horizontal_spacing=0.02,
)
for pos, idx in enumerate(indices):
    row, col = pos // 10 + 1, pos % 10 + 1
    img = sample_images[idx].squeeze().numpy()
    fig.add_trace(
        go.Heatmap(z=img[::-1], colorscale="Gray_r", showscale=False,
                   hovertemplate="pixel (%{x}, %{y}): %{z:.2f}<extra></extra>"),
        row=row, col=col,
    )
    fig.update_xaxes(showticklabels=False, row=row, col=col)
    fig.update_yaxes(showticklabels=False, row=row, col=col)
fig.update_layout(
    title_text="FashionMNIST \u2014 Sample Grid (2 per class)",
    height=320, width=900, margin=dict(t=60, b=10, l=10, r=10),
)
fig.show()

## 2. Architecture

We use **Adaptive Layer Normalization (adaLN)** — the class embedding modulates the scale and shift of every GroupNorm layer, injecting class information at every depth. This is the same mechanism DiT uses for timestep conditioning.

See [adaLN Conditioning](https://overfitting.club/posts/tutorials/deep_learning/class_conditional_vae/class_conditional_vae.html#adaln-conditioning-modulating-every-layer) for the full explanation and diagrams.

In [ ]:
#@title Building blocks: ResNetBlock, SelfAttention, AdaLNResNetBlock

class ResNetBlock(nn.Module):
    """Residual block: GroupNorm \u2192 SiLU \u2192 Conv \u2192 GroupNorm \u2192 SiLU \u2192 Conv + skip."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.GroupNorm(min(8, in_ch), in_ch),
            nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
        )
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.net(x) + self.skip(x)


class SelfAttention(nn.Module):
    """Single-head self-attention over spatial positions."""

    def __init__(self, ch: int):
        super().__init__()
        self.norm = nn.GroupNorm(min(8, ch), ch)
        self.q = nn.Conv2d(ch, ch, 1)
        self.k = nn.Conv2d(ch, ch, 1)
        self.v = nn.Conv2d(ch, ch, 1)
        self.proj = nn.Conv2d(ch, ch, 1)
        self.scale = ch ** -0.5

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        q = self.q(h).view(B, C, -1)
        k = self.k(h).view(B, C, -1)
        v = self.v(h).view(B, C, -1)
        attn = (q.transpose(1, 2) @ k) * self.scale
        attn = attn.softmax(dim=-1)
        out = (v @ attn.transpose(1, 2)).view(B, C, H, W)
        return x + self.proj(out)


class AdaLNResNetBlock(nn.Module):
    """ResNet block with adaptive layer normalization for class conditioning."""

    def __init__(self, in_ch: int, out_ch: int, cond_dim: int = 128):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(8, in_ch), in_ch, affine=False)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(min(8, out_ch), out_ch, affine=False)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.adaln_proj1 = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, 2 * in_ch))
        self.adaln_proj2 = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, 2 * out_ch))

    def forward(self, x, cond):
        gamma1, beta1 = self.adaln_proj1(cond).chunk(2, dim=1)
        gamma2, beta2 = self.adaln_proj2(cond).chunk(2, dim=1)
        gamma1 = gamma1[:, :, None, None]
        beta1  = beta1[:, :, None, None]
        gamma2 = gamma2[:, :, None, None]
        beta2  = beta2[:, :, None, None]

        h = self.norm1(x)
        h = (1 + gamma1) * h + beta1  # adaLN modulation
        h = F.silu(h)
        h = self.conv1(h)
        h = self.norm2(h)
        h = (1 + gamma2) * h + beta2
        h = F.silu(h)
        h = self.conv2(h)
        return h + self.skip(x)

In [ ]:
#@title adaLN cVAE model

class AdaLNEncoder(nn.Module):
    def __init__(self, z_channels=1, num_classes=NUM_CLASSES, cond_dim=COND_DIM):
        super().__init__()
        self.class_emb = nn.Embedding(num_classes, cond_dim)
        self.conv_in = nn.Conv2d(1, 32, 3, padding=1)
        self.block1a = AdaLNResNetBlock(32, 64, cond_dim)
        self.block1b = AdaLNResNetBlock(64, 64, cond_dim)
        self.down1 = nn.Conv2d(64, 64, 3, stride=2, padding=1)
        self.block2a = AdaLNResNetBlock(64, 128, cond_dim)
        self.block2b = AdaLNResNetBlock(128, 128, cond_dim)
        self.down2 = nn.Conv2d(128, 128, 3, stride=2, padding=1)
        self.mid1 = AdaLNResNetBlock(128, 128, cond_dim)
        self.mid_attn = SelfAttention(128)
        self.mid2 = AdaLNResNetBlock(128, 128, cond_dim)
        self.norm_out = nn.GroupNorm(8, 128)
        self.conv_out = nn.Conv2d(128, 2 * z_channels, 3, padding=1)

    def forward(self, x, labels):
        cond = self.class_emb(labels)
        h = self.conv_in(x)
        h = self.block1a(h, cond); h = self.block1b(h, cond); h = self.down1(h)
        h = self.block2a(h, cond); h = self.block2b(h, cond); h = self.down2(h)
        h = self.mid1(h, cond); h = self.mid_attn(h); h = self.mid2(h, cond)
        return self.conv_out(F.silu(self.norm_out(h)))


class AdaLNDecoder(nn.Module):
    def __init__(self, z_channels=1, num_classes=NUM_CLASSES, cond_dim=COND_DIM):
        super().__init__()
        self.class_emb = nn.Embedding(num_classes, cond_dim)
        self.conv_in = nn.Conv2d(z_channels, 128, 3, padding=1)
        self.mid1 = AdaLNResNetBlock(128, 128, cond_dim)
        self.mid_attn = SelfAttention(128)
        self.mid2 = AdaLNResNetBlock(128, 128, cond_dim)
        self.block2a = AdaLNResNetBlock(128, 128, cond_dim)
        self.block2b = AdaLNResNetBlock(128, 128, cond_dim)
        self.block2c = AdaLNResNetBlock(128, 64, cond_dim)
        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(64, 64, 3, padding=1))
        self.block1a = AdaLNResNetBlock(64, 64, cond_dim)
        self.block1b = AdaLNResNetBlock(64, 64, cond_dim)
        self.block1c = AdaLNResNetBlock(64, 32, cond_dim)
        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(32, 32, 3, padding=1))
        self.norm_out = nn.GroupNorm(8, 32)
        self.conv_out = nn.Conv2d(32, 1, 3, padding=1)

    def forward(self, z, labels):
        cond = self.class_emb(labels)
        h = self.conv_in(z)
        h = self.mid1(h, cond); h = self.mid_attn(h); h = self.mid2(h, cond)
        h = self.block2a(h, cond); h = self.block2b(h, cond); h = self.block2c(h, cond); h = self.up2(h)
        h = self.block1a(h, cond); h = self.block1b(h, cond); h = self.block1c(h, cond); h = self.up1(h)
        return torch.sigmoid(self.conv_out(F.silu(self.norm_out(h))))

    def decode_with_cond(self, z, cond):
        """Decode with a pre-computed conditioning vector (for class blending)."""
        h = self.conv_in(z)
        h = self.mid1(h, cond); h = self.mid_attn(h); h = self.mid2(h, cond)
        h = self.block2a(h, cond); h = self.block2b(h, cond); h = self.block2c(h, cond); h = self.up2(h)
        h = self.block1a(h, cond); h = self.block1b(h, cond); h = self.block1c(h, cond); h = self.up1(h)
        return torch.sigmoid(self.conv_out(F.silu(self.norm_out(h))))


class AdaLNCVAE(nn.Module):
    def __init__(self, z_channels=1, num_classes=NUM_CLASSES, cond_dim=COND_DIM):
        super().__init__()
        self.encoder = AdaLNEncoder(z_channels, num_classes, cond_dim)
        self.decoder = AdaLNDecoder(z_channels, num_classes, cond_dim)
        self.z_channels = z_channels

    def reparameterize(self, mu, log_var):
        return mu + torch.exp(0.5 * log_var) * torch.randn_like(mu)

    def forward(self, x, labels):
        h = self.encoder(x, labels)
        mu, log_var = h.chunk(2, dim=1)
        z = self.reparameterize(mu, log_var)
        return self.decoder(z, labels), mu, log_var, z

    @torch.no_grad()
    def generate(self, labels, device=None):
        device = device or next(self.parameters()).device
        labels = labels.to(device)
        z = torch.randn(labels.size(0), self.z_channels, 7, 7, device=device)
        return self.decoder(z, labels)


model = AdaLNCVAE(z_channels=Z_CHANNELS).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

t = Table(title="adaLN cVAE Architecture")
t.add_column("Component", style="cyan")
t.add_column("Detail", style="magenta")
t.add_column("Output Shape", style="green")
for name, detail, shape in [
    ("Encoder", "conv_in", "32×28×28"),
    ("", "2× adaLN-ResBlock(32→64) + ↓stride", "64×14×14"),
    ("", "2× adaLN-ResBlock(64→128) + ↓stride", "128×7×7"),
    ("", "adaLN-ResBlock + SelfAttn + adaLN-ResBlock", "128×7×7"),
    ("", "GN + SiLU + Conv → μ, log σ²", f"2×({Z_CHANNELS}×7×7)"),
    ("Latent", "Reparameterize", f"{Z_CHANNELS}×7×7 = {Z_CHANNELS * 49} values"),
    ("Decoder", "conv_in + mid + up2 + up1", "1×28×28"),
    ("Conditioning", "Embedding(10, 128) → adaLN at every block", "shared c (128,)"),
]:
    t.add_row(name, detail, shape)
console.print(t)
console.print(f"\n[bold]Total parameters:[/bold] {sum(p.numel() for p in model.parameters()):,}")

## 3. Training

The loss is the conditional ELBO: MSE reconstruction + β · KL divergence. With β=0.01 (higher than the unconditional VAE) we prioritize generation quality over reconstruction fidelity.

In [ ]:
#@title TensorBoard

%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
#@title Train adaLN cVAE

from torchvision.utils import make_grid

fixed_test_imgs = sample_images[:8].to(DEVICE)
fixed_test_lbls = sample_labels[:8].to(DEVICE)

# Fixed z for generation: same style across all 10 classes
fixed_z = torch.randn(1, Z_CHANNELS, 7, 7, device=DEVICE)

for epoch in range(EPOCHS):
    model.train()
    epoch_total, epoch_recon, epoch_kl = 0.0, 0.0, 0.0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        x_hat, mu, log_var, _ = model(images, labels)
        recon = criterion(x_hat, images)
        kl = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
        loss = recon + KL_WEIGHT * kl

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        b = images.size(0)
        epoch_total += loss.item() * b
        epoch_recon += recon.item() * b
        epoch_kl += kl.item() * b

    n = len(train_dataset)
    avg_total = epoch_total / n
    avg_recon = epoch_recon / n
    avg_kl = epoch_kl / n

    writer.add_scalar("Loss/total", avg_total, epoch)
    writer.add_scalar("Loss/reconstruction", avg_recon, epoch)
    writer.add_scalar("Loss/kl_divergence", avg_kl, epoch)

    if epoch == 0 or (epoch + 1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            # Reconstruction: original vs reconstructed
            recon_imgs, _, _, _ = model(fixed_test_imgs, fixed_test_lbls)
            comparison = torch.cat([fixed_test_imgs, recon_imgs])
            writer.add_image("Reconstructions", make_grid(comparison, nrow=8), epoch)

            # Generation: fixed z, one image per class (0-9)
            all_labels = torch.arange(NUM_CLASSES, device=DEVICE)
            z_repeated = fixed_z.expand(NUM_CLASSES, -1, -1, -1)
            generated = model.decoder(z_repeated, all_labels)
            writer.add_image("Generation/fixed_z_all_classes", make_grid(generated, nrow=NUM_CLASSES), epoch)

    if (epoch + 1) % 20 == 0:
        console.print(
            f"Epoch {epoch+1:3d}/{EPOCHS}  "
            f"total={avg_total:.6f}  recon={avg_recon:.6f}  kl={avg_kl:.4f}"
        )

writer.flush()
console.print("[bold green]Training complete![/bold green]")

## 4. Results

### Conditional Generation

Sample z ~ N(0, I) and specify the class — 8 random outputs per category.

In [ ]:
#@title Conditional generation grid

model.eval()
n_samples = 8

fig = make_subplots(
    rows=10, cols=n_samples,
    vertical_spacing=0.02, horizontal_spacing=0.02,
    row_titles=[CLASS_NAMES[c] for c in range(10)],
)

for c in range(10):
    labels = torch.full((n_samples,), c, dtype=torch.long)
    generated = model.generate(labels, device=DEVICE).cpu()
    for i in range(n_samples):
        img = generated[i].squeeze().numpy()
        fig.add_trace(
            go.Heatmap(z=img[::-1], colorscale="Gray_r", showscale=False,
                       hovertemplate="(%{x},%{y}): %{z:.2f}<extra></extra>"),
            row=c + 1, col=i + 1,
        )
        fig.update_xaxes(showticklabels=False, row=c + 1, col=i + 1)
        fig.update_yaxes(showticklabels=False, row=c + 1, col=i + 1)

fig.update_layout(
    title_text="adaLN cVAE \u2014 Conditional Generation (z ~ N(0, I), class specified)",
    height=120 * 10, width=900,
    margin=dict(t=40, b=10, l=100, r=10),
)
fig.show()

### Reconstruction Quality

In [ ]:
#@title Reconstruction comparison

model.eval()
with torch.no_grad():
    imgs = sample_images[:10].to(DEVICE)
    lbls = sample_labels[:10].to(DEVICE)
    recon, _, _, _ = model(imgs, lbls)
    recon = recon.cpu()

fig = make_subplots(
    rows=2, cols=10, vertical_spacing=0.02, horizontal_spacing=0.01,
    row_titles=["Original", "adaLN cVAE"],
)
for r, row_imgs in enumerate([sample_images[:10], recon]):
    for c in range(10):
        img = row_imgs[c].squeeze().numpy()
        fig.add_trace(
            go.Heatmap(z=img[::-1], colorscale="Gray_r", showscale=False,
                       hovertemplate="(%{x},%{y}): %{z:.2f}<extra></extra>"),
            row=r + 1, col=c + 1,
        )
        fig.update_xaxes(showticklabels=False, row=r + 1, col=c + 1)
        fig.update_yaxes(showticklabels=False, row=r + 1, col=c + 1)

fig.update_layout(
    title_text="Reconstructions \u2014 Original vs adaLN cVAE",
    height=250, width=900, margin=dict(t=40, b=10, l=80, r=10),
)
fig.show()

## 5. Latent Space

In [ ]:
#@title t-SNE projection

from sklearn.manifold import TSNE

model.eval()
all_mu, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        h = model.encoder(images.to(DEVICE), labels.to(DEVICE))
        mu, _ = h.chunk(2, dim=1)
        all_mu.append(mu.cpu().view(mu.size(0), -1))
        all_labels.append(labels)

all_mu = torch.cat(all_mu).numpy()
all_labels = torch.cat(all_labels).numpy()

# Subsample for speed
rng = np.random.default_rng(42)
sub_idx = np.concatenate([rng.choice(np.where(all_labels == c)[0], size=300, replace=False) for c in range(10)])

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
z_2d = tsne.fit_transform(all_mu[sub_idx])
labels_sub = all_labels[sub_idx]

PALETTE = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
           "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990"]

fig = go.Figure()
for c in range(10):
    mask = labels_sub == c
    fig.add_trace(go.Scattergl(
        x=z_2d[mask, 0], y=z_2d[mask, 1],
        mode="markers", marker=dict(size=3, color=PALETTE[c], opacity=0.6),
        name=CLASS_NAMES[c],
    ))
fig.update_layout(
    title="t-SNE of adaLN cVAE Latent Space",
    height=500, width=700, template="plotly_white",
    legend=dict(itemsizing="constant"),
)
fig.show()

## 6. Interactive Conditional Generation

Pick a class and generate samples. Use the **Resample** button to draw new z vectors.

In [ ]:
#@title Interactive generation

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import ipywidgets as widgets
from IPython.display import display

model.eval()
N_GEN = 8

class_dropdown = widgets.Dropdown(
    options={CLASS_NAMES[i]: i for i in range(10)}, value=7, description="Class:",
)
resample_btn = widgets.Button(description="Resample z", button_style="info", icon="random")

init_z = torch.randn(N_GEN, Z_CHANNELS, 7, 7, device=DEVICE)
state = {"z": init_z}


def make_gen_grid(z, class_idx):
    labels = torch.full((N_GEN,), class_idx, dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        imgs = model.decoder(z, labels).cpu()
    grid = np.concatenate([imgs[i].squeeze().numpy() for i in range(N_GEN)], axis=1)
    return grid


fig_gen = go.FigureWidget(
    data=[go.Heatmap(z=make_gen_grid(init_z, 7)[::-1], colorscale="Gray_r", showscale=False)],
    layout=dict(title=f"Generated: {CLASS_NAMES[7]}", height=200, width=700,
                xaxis=dict(showticklabels=False, scaleanchor="y"),
                yaxis=dict(showticklabels=False),
                margin=dict(t=40, b=10, l=10, r=10)),
)


def render(change=None):
    grid = make_gen_grid(state["z"], class_dropdown.value)
    with fig_gen.batch_update():
        fig_gen.data[0].z = grid[::-1]
        fig_gen.layout.title.text = f"Generated: {CLASS_NAMES[class_dropdown.value]}"


def on_resample(btn):
    state["z"] = torch.randn(N_GEN, Z_CHANNELS, 7, 7, device=DEVICE)
    render()


class_dropdown.observe(render, names="value")
resample_btn.on_click(on_resample)

display(widgets.VBox([widgets.HBox([class_dropdown, resample_btn]), fig_gen]))

## 7. Interactive Class Blending

Blend two class embeddings with a slider: `c_blend = (1-α)·c_A + α·c_B`. This creates hybrid garments that smoothly morph between categories.

In [ ]:
#@title Interactive class blending

model.eval()
N_BLEND = 4

class_a_dd = widgets.Dropdown(options={CLASS_NAMES[i]: i for i in range(10)}, value=7, description="Class A:")
class_b_dd = widgets.Dropdown(options={CLASS_NAMES[i]: i for i in range(10)}, value=9, description="Class B:")
alpha_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05, description="\u03b1:",
    continuous_update=True, style={"description_width": "30px"},
    layout=widgets.Layout(width="500px"),
)
blend_resample = widgets.Button(description="New z", button_style="info", icon="random")

blend_z = torch.randn(N_BLEND, Z_CHANNELS, 7, 7, device=DEVICE)
blend_state = {"z": blend_z}


def make_blend_grid():
    alpha = alpha_slider.value
    ca, cb = class_a_dd.value, class_b_dd.value
    emb = model.decoder.class_emb
    with torch.no_grad():
        c_a = emb(torch.tensor([ca], device=DEVICE))
        c_b = emb(torch.tensor([cb], device=DEVICE))
        c_blend = (1 - alpha) * c_a + alpha * c_b
        c_blend = c_blend.expand(N_BLEND, -1)
        imgs = model.decoder.decode_with_cond(blend_state["z"], c_blend).cpu()
    return np.concatenate([imgs[i].squeeze().numpy() for i in range(N_BLEND)], axis=1)


fig_blend = go.FigureWidget(
    data=[go.Heatmap(z=make_blend_grid()[::-1], colorscale="Gray_r", showscale=False)],
    layout=dict(title="Sneaker (50%) + Ankle boot (50%)", height=200, width=500,
                xaxis=dict(showticklabels=False, scaleanchor="y"),
                yaxis=dict(showticklabels=False),
                margin=dict(t=40, b=10, l=10, r=10)),
)


def render_blend(change=None):
    grid = make_blend_grid()
    alpha = alpha_slider.value
    with fig_blend.batch_update():
        fig_blend.data[0].z = grid[::-1]
        fig_blend.layout.title.text = (
            f"{CLASS_NAMES[class_a_dd.value]} ({100-int(alpha*100)}%) + "
            f"{CLASS_NAMES[class_b_dd.value]} ({int(alpha*100)}%)"
        )


def on_blend_resample(btn):
    blend_state["z"] = torch.randn(N_BLEND, Z_CHANNELS, 7, 7, device=DEVICE)
    render_blend()


class_a_dd.observe(render_blend, names="value")
class_b_dd.observe(render_blend, names="value")
alpha_slider.observe(render_blend, names="value")
blend_resample.on_click(on_blend_resample)

display(widgets.VBox([
    widgets.HBox([class_a_dd, class_b_dd, blend_resample]),
    alpha_slider,
    fig_blend,
]))

## 8. Interactive Latent Interpolation

Pick two classes and smoothly walk between two encoded samples in latent space while keeping the class label fixed.

In [ ]:
#@title Interactive intra-class interpolation

model.eval()
N_INTERP = 4

# Collect test images by class
class_pools = {c: [] for c in range(10)}
for images, labels in test_loader:
    for c in range(10):
        mask = (labels == c).nonzero(as_tuple=True)[0]
        for idx in mask:
            class_pools[c].append(images[idx])

interp_class_dd = widgets.Dropdown(
    options={CLASS_NAMES[i]: i for i in range(10)}, value=7, description="Class:",
)
t_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05, description="t:",
    continuous_update=True, style={"description_width": "30px"},
    layout=widgets.Layout(width="500px"),
)
interp_resample = widgets.Button(description="New pair", button_style="info", icon="random")

interp_state = {"mu_a": [], "mu_b": []}


def encode_pairs():
    c = interp_class_dd.value
    interp_state["mu_a"], interp_state["mu_b"] = [], []
    with torch.no_grad():
        idxs = np.random.choice(len(class_pools[c]), N_INTERP * 2, replace=False)
        for i in range(N_INTERP):
            lbl = torch.tensor([c], device=DEVICE)
            ha = model.encoder(class_pools[c][idxs[2*i]].unsqueeze(0).to(DEVICE), lbl)
            mu_a, _ = ha.chunk(2, dim=1)
            interp_state["mu_a"].append(mu_a)
            hb = model.encoder(class_pools[c][idxs[2*i+1]].unsqueeze(0).to(DEVICE), lbl)
            mu_b, _ = hb.chunk(2, dim=1)
            interp_state["mu_b"].append(mu_b)


def make_interp_grid(t_val):
    c = interp_class_dd.value
    lbl = torch.tensor([c], device=DEVICE)
    imgs = []
    with torch.no_grad():
        for i in range(N_INTERP):
            z_t = (1 - t_val) * interp_state["mu_a"][i] + t_val * interp_state["mu_b"][i]
            img = model.decoder(z_t, lbl).squeeze().cpu().numpy()
            imgs.append(img)
    return np.concatenate(imgs, axis=1)


encode_pairs()

fig_interp = go.FigureWidget(
    data=[go.Heatmap(z=make_interp_grid(0.5)[::-1], colorscale="Gray_r", showscale=False)],
    layout=dict(title=f"{CLASS_NAMES[7]} interpolation (t=0.50)", height=200, width=500,
                xaxis=dict(showticklabels=False, scaleanchor="y"),
                yaxis=dict(showticklabels=False),
                margin=dict(t=40, b=10, l=10, r=10)),
)


def render_interp(change=None):
    t = t_slider.value
    grid = make_interp_grid(t)
    with fig_interp.batch_update():
        fig_interp.data[0].z = grid[::-1]
        fig_interp.layout.title.text = f"{CLASS_NAMES[interp_class_dd.value]} interpolation (t={t:.2f})"


def on_interp_resample(btn):
    encode_pairs()
    render_interp()


def on_interp_class(change):
    encode_pairs()
    render_interp()


interp_class_dd.observe(on_interp_class, names="value")
t_slider.observe(render_interp, names="value")
interp_resample.on_click(on_interp_resample)

display(widgets.VBox([
    widgets.HBox([interp_class_dd, interp_resample]),
    t_slider,
    fig_interp,
]))

## 9. Disentanglement

### Fix z, Vary Class

Each column uses the same z — only the class changes. Consistent visual style across rows = disentanglement.

In [ ]:
#@title Disentanglement: fix z, vary class

model.eval()
n_z = 8
torch.manual_seed(123)
fixed_z = torch.randn(n_z, Z_CHANNELS, 7, 7, device=DEVICE)

fig = make_subplots(
    rows=10, cols=n_z,
    vertical_spacing=0.02, horizontal_spacing=0.02,
    row_titles=[CLASS_NAMES[c] for c in range(10)],
)

with torch.no_grad():
    for c in range(10):
        labels = torch.full((n_z,), c, dtype=torch.long, device=DEVICE)
        generated = model.decoder(fixed_z, labels).cpu()
        for i in range(n_z):
            img = generated[i].squeeze().numpy()
            fig.add_trace(
                go.Heatmap(z=img[::-1], colorscale="Gray_r", showscale=False),
                row=c + 1, col=i + 1,
            )
            fig.update_xaxes(showticklabels=False, row=c + 1, col=i + 1)
            fig.update_yaxes(showticklabels=False, row=c + 1, col=i + 1)

fig.update_layout(
    title_text="Fix z, Vary Class \u2014 Same Style Across Classes",
    height=120 * 10, width=900,
    margin=dict(t=40, b=10, l=100, r=10),
)
fig.show()

### Fix Class, Vary z

Each row is a single class with different z vectors. Diversity within rows = the model avoids posterior collapse.

In [ ]:
#@title Disentanglement: fix class, vary z

fig2 = make_subplots(
    rows=10, cols=n_z,
    vertical_spacing=0.02, horizontal_spacing=0.02,
    row_titles=[CLASS_NAMES[c] for c in range(10)],
)

with torch.no_grad():
    for c in range(10):
        labels = torch.full((n_z,), c, dtype=torch.long, device=DEVICE)
        z_varied = torch.randn(n_z, Z_CHANNELS, 7, 7, device=DEVICE)
        generated = model.decoder(z_varied, labels).cpu()
        for i in range(n_z):
            img = generated[i].squeeze().numpy()
            fig2.add_trace(
                go.Heatmap(z=img[::-1], colorscale="Gray_r", showscale=False),
                row=c + 1, col=i + 1,
            )
            fig2.update_xaxes(showticklabels=False, row=c + 1, col=i + 1)
            fig2.update_yaxes(showticklabels=False, row=c + 1, col=i + 1)

fig2.update_layout(
    title_text="Fix Class, Vary z \u2014 Style Variation Within Each Class",
    height=120 * 10, width=900,
    margin=dict(t=40, b=10, l=100, r=10),
)
fig2.show()

### Class Blending Grid

Interpolate between pairs of semantically compatible classes at 10% increments.

In [ ]:
#@title Class blending grid

blend_pairs = [
    (7, 9, "Sneaker \u2192 Ankle boot"),
    (5, 7, "Sandal \u2192 Sneaker"),
    (0, 6, "T-shirt \u2192 Shirt"),
    (2, 4, "Pullover \u2192 Coat"),
    (1, 3, "Trouser \u2192 Dress"),
]

n_steps = 11
alphas = np.linspace(0, 1, n_steps)

fig = make_subplots(
    rows=len(blend_pairs), cols=n_steps,
    vertical_spacing=0.04, horizontal_spacing=0.01,
    row_titles=[p[2] for p in blend_pairs],
    column_titles=[f"{int(a*100)}%" for a in alphas],
)

torch.manual_seed(42)
blend_fixed_z = torch.randn(1, Z_CHANNELS, 7, 7, device=DEVICE)

with torch.no_grad():
    emb = model.decoder.class_emb
    for row_idx, (ca, cb, _) in enumerate(blend_pairs):
        c_a = emb(torch.tensor([ca], device=DEVICE))
        c_b = emb(torch.tensor([cb], device=DEVICE))
        for col_idx, alpha in enumerate(alphas):
            c_blend = (1 - alpha) * c_a + alpha * c_b
            img = model.decoder.decode_with_cond(blend_fixed_z, c_blend).cpu().squeeze().numpy()
            fig.add_trace(
                go.Heatmap(z=img[::-1], colorscale="Gray_r", showscale=False),
                row=row_idx + 1, col=col_idx + 1,
            )
            fig.update_xaxes(showticklabels=False, row=row_idx + 1, col=col_idx + 1)
            fig.update_yaxes(showticklabels=False, row=row_idx + 1, col=col_idx + 1)

fig.update_layout(
    title_text="Class Blending \u2014 Interpolating Between Category Embeddings",
    height=160 * len(blend_pairs), width=950,
    margin=dict(t=60, b=10, l=120, r=10),
)
fig.update_annotations(font_size=9)
fig.show()

---

**Full tutorial:** [Class-Conditional VAE](https://overfitting.club/posts/tutorials/deep_learning/class_conditional_vae/class_conditional_vae.html) | **Blog:** [Overfitting Club](https://overfitting.club)